In [32]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
import numpy as np

In [ ]:
root_pkl = Path("train/ensemble_inference/lesions").glob("*.pkl")
dfs = []
for pkl in tqdm(root_pkl):
    df = pd.read_pickle(pkl)
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)


0it [00:00, ?it/s]

In [21]:
combined.head()

,image_id,lesion_id,image_path,area,centroid,bbox,coordinates,contours
0,007-2920-100,MICROANEURYSMS,ensemble_inference/segmentations/007-2920-100.png,62.0,"(504.06451612903226, 845.3709677419355)","(500, 841, 509, 850)","[[500, 844], [500, 845], [500, 846], [500, 847...","[[[508.5, 847.0], [508.5, 846.0], [508.5, 845...."
1,007-2920-100,EXUDATES,ensemble_inference/segmentations/007-2920-100.png,62.0,"(557.0483870967741, 854.9032258064516)","(553, 851, 562, 860)","[[553, 853], [553, 854], [553, 855], [554, 852...","[[[561.5, 856.0], [561.5, 855.0], [561.5, 854...."
2,007-2920-100,EXUDATES,ensemble_inference/segmentations/007-2920-100.png,114.0,"(642.1491228070175, 836.0087719298245)","(636, 831, 649, 842)","[[636, 834], [636, 835], [636, 836], [636, 837...","[[[648.5, 838.0], [648.5, 837.0], [648.5, 836...."
3,007-2920-100,MICROANEURYSMS,ensemble_inference/segmentations/007-2920-100.png,6.0,"(661.8333333333334, 661.8333333333334)","(661, 661, 664, 664)","[[661, 661], [661, 662], [662, 661], [662, 662...","[[[663.5, 662.0], [663.0, 661.5], [662.5, 661...."
4,007-2920-100,MICROANEURYSMS,ensemble_inference/segmentations/007-2920-100.png,57.0,"(709.4561403508771, 601.140350877193)","(706, 597, 714, 606)","[[706, 599], [706, 600], [706, 601], [706, 602...","[[[713.5, 604.0], [713.5, 603.0], [713.5, 602...."


In [43]:
gt = pd.read_csv(
    "/home/clement/Documents/data/DDR-dataset/DR_grading/test.txt",
    header=None,
    sep=" ",
    names=["image_id", "label"],
)
xs = []
ys = []
all_lesions = combined["lesion_id"].unique()
# We create a mapping from lesion_id to an integer index
lesion_mapping = {lesion_id: idx for idx, lesion_id in enumerate(all_lesions)}
for image_id in tqdm(combined["image_id"].unique()):
    # We count the number of lesions for each image
    count = combined[combined["image_id"] == image_id]["lesion_id"].value_counts()
    x = np.asarray([count.get(lesion_id, 0) for lesion_id in all_lesions])
    xs.append(x)
    y = gt[gt["image_id"] == image_id + ".jpg"]["label"].values[0]
    ys.append(y)


  0%|          | 0/2747 [00:00<?, ?it/s]

In [54]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import cohen_kappa_score, make_scorer, accuracy_score


xs = np.array(xs)
ys = np.array(ys)
Y = ys
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42)
results = {}
cv_auc = cross_val_score(
    lr, xs, Y, cv=cv, scoring=make_scorer(cohen_kappa_score, weights="quadratic")
)
results["All lesions (LR, 5-fold CV)"] = {
    "AUC-ROC mean": cv_auc.mean(),
    "AUC-ROC std": cv_auc.std(),
}
results

{'All lesions (LR, 5-fold CV)': {'AUC-ROC mean': 0.6722539249425121,
  'AUC-ROC std': 0.025480491831913494}}